In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import sys
import types
import numpy
import os
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report)
print(os.getcwd())
#sys.path.append("//mnt/batch/tasks/shared/LS_root/mounts/clusters/zahracpu/code/Users/zahra.sobhaninia/MODELS/Corebehrt_OOT")
sys.path.append("//mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/MODELS/Corebehrt_OOT")
import corebehrt

/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/MODELS/Corebehrt_OOT/Results_032026


In [6]:
import pandas as pd
from azureml.core import Workspace, Dataset, Datastore
import pandas as pd
import torch
import io

subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'
workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "researcher_data")

dataset_outcomes = Dataset.File.from_files(
    path=(datastore, 'Zahra/032026/CoreBehrt_OOT/MD/2_Finetune/ARF/Data/Finetunedata/outcomes.csv')
)
local_path = dataset_outcomes.download(target_path=".", overwrite=True)[0]
df = pd.read_csv(local_path)

print(df.columns.tolist())
print(df.shape)
print(df.iloc[:, -1].value_counts())

{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
['subject_id', 'time', 'abspos']
(431, 3)
440184    5
442344    4
442584    4
445368    4
447360    4
         ..
440592    1
445032    1
445416    1
445080    1
439344    1
Name: abspos, Length: 280, dtype: int64


In [7]:
print(df.columns.tolist())
print(df.head(10))

['subject_id', 'time', 'abspos']
   subject_id        time  abspos
0     1397883  2020-02-14  439344
1     1902274  2020-07-08  442824
2     1465833  2020-02-22  439536
3      350583  2020-03-25  440304
4      610688  2020-02-21  439512
5      719046  2020-01-02  438312
6     1027124  2020-06-04  442008
7      449818  2020-01-10  438504
8      681181  2020-06-03  441984
9     1461734  2020-02-29  439704


In [9]:
dataset_outcomes = Dataset.File.from_files(
    path=(datastore, 'Zahra/022026/CoreBehrt_CV/MDPS/2_Finetune/ARF/Data/Processed_data/outcomes.csv')
)
local_path = dataset_outcomes.download(target_path=".", overwrite=True)[0]
df2 = pd.read_csv(local_path)

print(df2.columns.tolist())
print(df2.shape)
print(df2.iloc[:, -1].value_counts())

{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
['subject_id', 'time', 'abspos']
(176, 3)
432312    2
433656    2
420504    2
423552    2
417336    2
         ..
424920    1
425040    1
425352    1
425616    1
411456    1
Name: abspos, Length: 167, dtype: int64


In [10]:
print(df2.columns.tolist())
print(df2.head(10))

['subject_id', 'time', 'abspos']
   subject_id        time  abspos
0      741296  2016-12-06  411384
1     2071755  2016-12-20  411720
2     1285502  2017-01-08  412176
3     1148563  2017-03-05  413520
4     1556138  2017-02-05  412848
5     1469711  2017-03-31  414144
6     1440199  2017-03-06  413544
7      831139  2017-03-24  413976
8      949880  2017-03-17  413808
9     1849203  2017-05-21  415368


In [12]:
import pandas as pd

# outcomes قدیمی (176 بیمار)
dataset_old = Dataset.File.from_files(
    path=(datastore, 'Zahra/022026/CoreBehrt_CV/MDPS/2_Finetune/ARF/Data/Processed_data/outcomes.csv')
)
local_old = dataset_old.download(target_path="./old", overwrite=True)[0]
df_old = pd.read_csv(local_old)

# outcomes جدید (431 بیمار)
dataset_new = Dataset.File.from_files(
    path=(datastore, 'Zahra/032026/CoreBehrt_OOT/MDPS/2_Finetune/ARF/Data/Finetunedata/outcomes.csv')
)
local_new = dataset_new.download(target_path="./new", overwrite=True)[0]
df_new = pd.read_csv(local_new)

# مقایسه
print("=== قدیمی ===")
print(f"تعداد بیمار: {df_old['subject_id'].nunique()}")
print(f"شکل داده: {df_old.shape}")
print(df_old.head())

print("\n=== جدید ===")
print(f"تعداد بیمار: {df_new['subject_id'].nunique()}")
print(f"شکل داده: {df_new.shape}")
print(df_new.head())

# بیماران مشترک
common = set(df_old['subject_id']) & set(df_new['subject_id'])
print(f"\nبیماران مشترک: {len(common)}")
print(f"فقط در قدیمی: {len(set(df_old['subject_id']) - set(df_new['subject_id']))}")
print(f"فقط در جدید: {len(set(df_new['subject_id']) - set(df_old['subject_id']))}")

{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
=== قدیمی ===
تعداد بیمار: 169
شکل داده: (176, 3)
   subject_id        time  abspos
0      741296  2016-12-06  411384
1     2071755  2016-12-20  411720
2     1285502  2017-01-08  412176
3     1148563  2017-03-05  413520
4     1556138  2017-02-05  412848

=== جدید ===
تعداد بیمار: 416
شکل داده: (431, 3)
   subject_id        time  abspos
0     1397883  2020-02-14  439344
1     1902274  2020-07-08  442824
2     1465833  2020-02-22  439536
3      350583  2020-03-25  440304
4      610688  2020-02-21  439512

بیماران مشترک: 37
فقط در قدیمی: 132
فقط در جدید: 379


In [13]:
# ببینیم زمان داده‌ها چقدر فرق دارن
df_old['time'] = pd.to_datetime(df_old['time'])
df_new['time'] = pd.to_datetime(df_new['time'])

print("=== قدیمی ===")
print(f"از: {df_old['time'].min()}")
print(f"تا: {df_old['time'].max()}")

print("\n=== جدید (OOT) ===")
print(f"از: {df_new['time'].min()}")
print(f"تا: {df_new['time'].max()}")

=== قدیمی ===
از: 2016-12-06 00:00:00
تا: 2022-01-08 00:00:00

=== جدید (OOT) ===
از: 2020-01-02 00:00:00
تا: 2021-05-30 00:00:00


In [14]:
# ببینیم index_dates هر دو چی میگه
dataset_idx_old = Dataset.File.from_files(
    path=(datastore, 'Zahra/022026/CoreBehrt_CV/MDPS/2_Finetune/ARF/Data/SelectCohort/index_dates.csv'))
local_idx_old = dataset_idx_old.download(target_path="./old", overwrite=True)[0]
df_idx_old = pd.read_csv(local_idx_old)

dataset_idx_new = Dataset.File.from_files(
    path=(datastore, 'Zahra/032026/CoreBehrt_OOT/MD/2_Finetune/ARF/Data/SelectCohort/index_dates.csv')
)
local_idx_new = dataset_idx_new.download(target_path="./new", overwrite=True)[0]
df_idx_new = pd.read_csv(local_idx_new)

print("=== قدیمی ===")
print(df_idx_old.shape)
print(df_idx_old.head())

print("\n=== جدید ===")
print(df_idx_new.shape)
print(df_idx_new.head())

{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
=== قدیمی ===
(52216, 2)
   subject_id        time
0          26  2019-02-15
1          49  2022-03-11
2         174  2017-08-24
3         242  2020-01-29
4         243  2020-05-12

=== جدید ===
(180947, 2)
   subject_id        time
0           5  2020-01-27
1          23  2020-06-19
2          39  2020-06-30
3          40  2020-07-07
4          44  2020-05-29


In [15]:
# ببینیم subject_id های outcome در index_dates هستن؟
old_outcome_ids = set(df_old['subject_id'])
new_outcome_ids = set(df_new['subject_id'])

old_index_ids = set(df_idx_old['subject_id'])
new_index_ids = set(df_idx_new['subject_id'])

print("=== قدیمی ===")
print(f"outcome IDs که در index_dates نیستن: {len(old_outcome_ids - old_index_ids)}")
print(f"نسبت positive: {len(old_outcome_ids)/len(old_index_ids)*100:.2f}%")

print("\n=== جدید ===")
print(f"outcome IDs که در index_dates نیستن: {len(new_outcome_ids - new_index_ids)}")
print(f"نسبت positive: {len(new_outcome_ids)/len(new_index_ids)*100:.2f}%")

=== قدیمی ===
outcome IDs که در index_dates نیستن: 0
نسبت positive: 0.32%

=== جدید ===
outcome IDs که در index_dates نیستن: 0
نسبت positive: 0.23%


In [16]:
# ببینیم subject_id های outcome در index_dates هستن؟
old_outcome_ids = set(df_old['subject_id'])
new_outcome_ids = set(df_new['subject_id'])

old_index_ids = set(df_idx_old['subject_id'])
new_index_ids = set(df_idx_new['subject_id'])

print("=== قدیمی ===")
print(f"outcome IDs که در index_dates نیستن: {len(old_outcome_ids - old_index_ids)}")
print(f"نسبت positive: {len(old_outcome_ids)/len(old_index_ids)*100:.2f}%")

print("\n=== جدید ===")
print(f"outcome IDs که در index_dates نیستن: {len(new_outcome_ids - new_index_ids)}")
print(f"نسبت positive: {len(new_outcome_ids)/len(new_index_ids)*100:.2f}%")

=== قدیمی ===
outcome IDs که در index_dates نیستن: 0
نسبت positive: 0.32%

=== جدید ===
outcome IDs که در index_dates نیستن: 0
نسبت positive: 0.23%


In [17]:
from azureml.core import Workspace, Dataset, Datastore

subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'
workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "researcher_data")

# لود exposure
dataset = Dataset.Tabular.from_delimited_files(
    path=(datastore, 'Zahra/032026/CoreBehrt_OOT/CreateOutcome_Finetune/ARF/exposure.csv')
)
exposure_df = dataset.to_pandas_dataframe()

print(f"Total records: {len(exposure_df)}")
print(f"Total unique patients: {exposure_df['subject_id'].nunique()}")

# چند بیمار بیش از یه جراحی دارن
surgery_count = exposure_df.groupby('subject_id')['time'].count()

print(f"\nPatients with 1 surgery: {(surgery_count == 1).sum()}")
print(f"Patients with 2+ surgeries: {(surgery_count > 1).sum()}")
print(f"Max surgeries per patient: {surgery_count.max()}")

# چند تا جراحی از دست میدیم
total_surgeries = len(exposure_df)
kept_surgeries = exposure_df['subject_id'].nunique()
lost_surgeries = total_surgeries - kept_surgeries

print(f"\nTotal surgeries: {total_surgeries}")
print(f"Kept (first only): {kept_surgeries}")
print(f"Lost: {lost_surgeries} ({lost_surgeries/total_surgeries*100:.1f}%)")

Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Total records: 211570
Total unique patients: 180947

Patients with 1 surgery: 156050
Patients with 2+ surgeries: 24897
Max surgeries per patient: 10

Total surgeries: 211570
Kept (first only): 180947
Lost: 30623 (14.5%)
